# 05 · 凍結 DINOv2 和街景排序器

讀取已鎖定的 2018–2022 年圖庫。先測小批流程，再提取完整圖庫和 query 的 embedding，量 Top‑100 能提供多少 25 km 內的候選。

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import importlib.metadata as metadata
import json
import subprocess
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/ranker_preprocessing/since_2018/dataset.json").is_file()), None)
if root is None:
    root = Path("/content/geoguessr-ranker-2018")
    root.mkdir(exist_ok=True)
    with ZipFile("/content/ranker_training_upload.zip") as archive:
        for member in archive.namelist():
            if not (root / member).resolve().is_relative_to(root.resolve()):
                raise ValueError(member)
        archive.extractall(root)

def run_script(script, *args):
    command = [sys.executable, "-B", "-u", str(root / "scripts" / script), *map(str, args)]
    with subprocess.Popen(command, cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

data = root / "data/ranker_preprocessing/since_2018"
output = root / "results/ranker_preprocessing/since_2018/25000"
lock = json.loads((data / "dataset.json").read_text())
assert lock["min_capture_year"] == 2018 and lock["manifests"]["gallery"]["n"] == 25_000
print({key: entry["n"] for key, entry in lock["manifests"].items()})
cached_summaries = (output / "smoke/pair_summary.json", output / "pair_summary.json", output / "ranker_summary.json")
if all(path.is_file() for path in cached_summaries):
    print("Reading verified results saved from Colab T4.")
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Connect the notebook to a Colab GPU runtime for image encoding.")
    print(torch.cuda.get_device_name(0))
    try:
        transformers_ok = metadata.version("transformers") == "4.57.6"
    except metadata.PackageNotFoundError:
        transformers_ok = False
    if not transformers_ok:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6"], check=True)


{'gallery': 25000, 'train': 12000, 'validation': 2000, 'coverage': 6000}
Reading verified results saved from Colab T4.


In [ ]:
common = ("--data-dir", data, "--gallery", 25_000, "--encode")
if not (output / "smoke/pair_summary.json").is_file():
    run_script("materialize_ranker_data.py", *common, "--smoke")
print((output / "smoke/pair_summary.json").read_text())

{
  "gallery_n": 256,
  "raw_k": 100,
  "hit_km": 25.0,
  "encoder_revision": "ed25f3a31f01632728cabb09d1542f84ab7b0056",
  "device": "cuda",
  "duplicates_removed": {
    "gallery": 0,
    "train": 0,
    "validation": 0
  },
  "splits": {
    "train": {
      "query_n": 32,
      "gallery_within_25": 2,
      "top100_hits": 2,
      "top1_hits": 0,
      "pair_query_n": 2,
      "pair_n": 16
    },
    "validation": {
      "query_n": 32,
      "gallery_within_25": 1,
      "top100_hits": 1,
      "top1_hits": 0,
      "pair_query_n": 1,
      "pair_n": 8
    }
  }
}



In [ ]:
if not (output / "pair_summary.json").is_file():
    run_script("materialize_ranker_data.py", *common)
summary = json.loads((output / "pair_summary.json").read_text())
print(json.dumps(summary, indent=2))

{
  "gallery_n": 25000,
  "raw_k": 100,
  "hit_km": 25.0,
  "encoder_revision": "ed25f3a31f01632728cabb09d1542f84ab7b0056",
  "device": "cuda",
  "duplicates_removed": {
    "gallery": 0,
    "train": 0,
    "validation": 0
  },
  "splits": {
    "train": {
      "query_n": 12000,
      "gallery_within_25": 11612,
      "top100_hits": 5923,
      "top1_hits": 1705,
      "pair_query_n": 5923,
      "pair_n": 47384
    },
    "validation": {
      "query_n": 2000,
      "gallery_within_25": 1915,
      "top100_hits": 983,
      "top1_hits": 290,
      "pair_query_n": 983,
      "pair_n": 7864
    }
  }
}


In [ ]:
if root == Path("/content/geoguessr-ranker-2018"):
    archive_path = Path("/content/ranker-2018-pairs.zip")
    with ZipFile(archive_path, "w", ZIP_DEFLATED, compresslevel=2) as archive:
        for name in ("pair_summary.json", "gallery_images.jsonl", "train_images.jsonl", "validation_images.jsonl",
                     "gallery_features.npz", "train_features.npz", "validation_features.npz",
                     "train_pairs.npy", "validation_pairs.npy", "train_candidates.jsonl", "validation_candidates.jsonl",
                     "train_coverage_by_country.csv", "validation_coverage_by_country.csv"):
            archive.write(output / name, name)
    print(archive_path)

In [ ]:
if not (output / "ranker_summary.json").is_file():
    run_script("train_ranker.py", output)
ranker = json.loads((output / "ranker_summary.json").read_text())
if ranker["status"] == "trained":
    print(f"Checkpoint selected on internal train split: epoch {ranker['best_epoch']}")
    if ranker["best_epoch"] == 0:
        print("The comparison uses the original cosine checkpoint because no trained epoch improved internal tuning.")
    for row in ranker["tuning_history"]:
        print(f"  Epoch {row['epoch']} · internal Top-1 at 25 km {row['hits_25']}/{row['n']}")
    for name in ("baseline", "reranked"):
        metrics = ranker[name]
        print(f"{name:8} | 25 km {metrics['hits_25']:,}/{metrics['n']:,} ({metrics['hit_rate_25']:.2%})"
              f" | median {metrics['median_km']:,.1f} km")
    print(f"Corrected: {ranker['improved']} · regressed: {ranker['regressed']}")
else:
    print(f"Usable train queries: {ranker['pair_queries']:,}; planned target {ranker['target']:,}.")

Checkpoint selected on internal train split: epoch 0
The comparison uses the original cosine checkpoint because no trained epoch improved internal tuning.
  Epoch 0 · internal Top-1 at 25 km 152/1200
  Epoch 1 · internal Top-1 at 25 km 8/1200
  Epoch 2 · internal Top-1 at 25 km 9/1200
  Epoch 3 · internal Top-1 at 25 km 11/1200
baseline | 25 km 290/2,000 (14.50%) | median 1,252.1 km
reranked | 25 km 290/2,000 (14.50%) | median 1,252.1 km
Corrected: 0 · regressed: 0


In [ ]:
if root == Path("/content/geoguessr-ranker-2018"):
    archive_path = Path("/content/ranker-2018-evaluation.zip")
    with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
        for name in ("ranker_summary.json", "ranker_predictions.jsonl", "ranker.pt", "pair_summary.json"):
            path = output / name
            if path.is_file():
                archive.write(path, name)
        archive.write(output / "smoke/pair_summary.json", "smoke_pair_summary.json")
    print(archive_path)